In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Cargar la base de datos
data_path = 'input/Production_Crops_Livestock_E_All_Data.csv'
filtered_data = pd.read_csv(data_path, encoding='latin1')


# Filtrar las columnas que contienen datos de rendimiento desde el año 2000 al 2022
year_columns = [f'Y{year}' for year in range(1992, 2022)] 
flag_columns = [f'Y{year}F' for year in range(1992, 2022)] 
filtered_data = filtered_data[['Area', 'Area Code (M49)', 'Item', 'Element'] + year_columns+ flag_columns]


# Cargar la tabla de grupos de ítems (para quedarme solo con los que representan cultivos)
item_groups = pd.read_csv('input/FAOSTAT_data_9-14-2023.csv')

# Seleccionar los grupos deseados
desired_groups = ["Crops, primary", "Cereals, primary", "Citrus Fruit, Total",                   "Fibre Crops Primary", "Fruit Primary", "Vegetables Primary"]

# Filtrar los ítems que pertenecen a los grupos deseados
filtered_items = item_groups[item_groups['Item Group'].isin(desired_groups)]

# Eliminar los ítems cuyo nombre coincide con el nombre del grupo
filtered_items = filtered_items[filtered_items['Item'] != filtered_items['Item Group']]

# Filtrar 'filtered_data' para quedarnos solo con los ítems que están en 'filtered_items'
filtered_data = filtered_data[filtered_data['Item'].isin(filtered_items['Item'])]


# 1. Filtrar el conjunto de datos para incluir solo las filas con "Yield" en la columna "Element".
yield_data = filtered_data[filtered_data['Element'] == 'Yield']

# 2. Agrupar por país y año y verificar si hay algún dato (no NA) para ese año.
valid_countries = yield_data.groupby('Area').apply(lambda group: all(group[year_columns].notna().any())).reset_index()
valid_countries.columns = ['Area', 'Valid']

# 3. Filtrar aquellos países que tienen datos para todos los años en el rango especificado.
valid_countries_list = valid_countries[valid_countries['Valid']]['Area'].tolist()
filtered_data = filtered_data[filtered_data['Area'].isin(valid_countries_list)]
# Verificar el resultado
#filtered_data.head()
#num_countries = filtered_data['Area'].nunique()
#num_countries

filtered_data.head()


In [ ]:
# Lista de códigos que deben ser eliminados
codes_to_remove = [
    "'001", "'002", "'014'", "'017", "'015", "'018", "'011", "'019", "'021", "'013", "'029'", "'005",
   "'142", "'143", "'030", "'034'", "'035", "'145", "'150", "'151", "'154", "'039", "'155", "'009",
    "'053", "'054", "'057", "'061", "'097","'199", "'432", "'722", "'901", "'902"
]

# Filtrar el dataframe para eliminar las entradas con los códigos especificados
filtered_data = filtered_data[~filtered_data['Area Code (M49)'].isin(codes_to_remove)]

# Cargar la tabla de códigos de países
country_codes = pd.read_csv('input/country_code.csv')

# Remover la comilla inicial y convertir a entero
filtered_data ['Area Code (M49)'] = filtered_data ['Area Code (M49)'].str.replace("'", '').astype(int)

# Unir la tabla de códigos de países con filtered_data usando 'Numeric code' y 'Area Code (M49)'
filtered_data = filtered_data.merge(country_codes[['Alpha-2code', 'Alpha-3code', 'Numericcode']],  left_on='Area Code (M49)', right_on='Numericcode', how='left')




In [ ]:
import pandas as pd
from scipy import stats

yield_data = filtered_data[filtered_data['Element'] == 'Yield']


# Sumar la producción por país y por año
yield_sum = yield_data.groupby(['Area', 'Area Code (M49)', 'Alpha-2code', 'Alpha-3code'])[year_columns].sum()
yield_avg = yield_sum.mean(axis=1)

################################ CREO LA BASE DE DATOS DONDE VOY A AÑADIR COLUMNAS
final_df = yield_sum.reset_index()[['Area', 'Area Code (M49)', 'Alpha-2code', 'Alpha-3code']]

# Función para contar las caídas por debajo del promedio
def count_falls(row, avg):
    return sum(value < avg for value in row)

# Aplicar la función para contar las caídas por debajo del promedio
falls_below_avg = yield_sum.apply(lambda x: count_falls(x, yield_avg[x.name]), axis=1)

# Calcular el desvío estándar y la tendencia de la producción para cada país
yield_std = yield_sum.std(axis=1)/yield_avg 
yield_trend = yield_sum.apply(lambda x: stats.linregress(range(len(year_columns)), x)[0], axis=1)/yield_avg 

# Calcular la diferencia interanual y contar caídas
yield_diff = yield_sum.diff(axis=1)
falls_interannual = (yield_diff < 0).sum(axis=1)

# Función para calcular la proporción de valores que no son 'A' ni 'X' para cada país y año
def calculate_non_ax_proportion(row):
    non_ax_count = sum(value not in ['A', 'X', ''] for value in row)
    total_count = len(row)
    return non_ax_count / total_count if total_count > 0 else 0

# Aplicar la función a cada fila de las columnas de flags y calcular el promedio por país
non_ax_proportions = yield_data[flag_columns].apply(calculate_non_ax_proportion, axis=1)
yield_data['non_ax_proportion'] = non_ax_proportions
non_ax_proportions_avg = yield_data.groupby('Area')['non_ax_proportion'].mean()
flags_proportion_avg = non_ax_proportions_avg.to_frame(name='yield_flags_proportion')



# Crear un DataFrame con las columnas deseadas

final_df['falls_below_avg'] = falls_below_avg.values
final_df['falls_interannual'] = falls_interannual.values
final_df['yield_std'] = yield_std.values
final_df['yield_trend'] = yield_trend.values
final_df['yield_trend&std'] = final_df['yield_trend'] * final_df['yield_std']

final_df = final_df.merge(flags_proportion_avg, left_on='Area', right_index=True, how='left')

final_df

In [ ]:
################## Cargar la base de datos PRODUCCION EN DOLARES INTERNACIONALES

data_path2 = 'input/Value_of_Production_E_All_Data.csv'
filtered_data_value = pd.read_csv(data_path2, encoding='latin1')

filtered_data_value ['Area Code (M49)'] = filtered_data_value ['Area Code (M49)'].str.replace("'", '').astype(int)

# Filtrar las columnas que contienen datos de rendimiento desde el año 2000 al 2022
filtered_data_value = filtered_data_value [['Area', 'Area Code (M49)', 'Item', 'Element'] + year_columns+ flag_columns]

# Eliminar los ítems cuyo nombre coincide con el nombre del grupo
filtered_items_value = filtered_items[filtered_items['Item'] != filtered_items['Item Group']]

# Filtrar 'filtered_data' para quedarnos solo con los ítems que están en 'filtered_items'
filtered_data_value  = filtered_data_value [filtered_data_value ['Item'].isin(filtered_items['Item'])]

filtered_data_value = filtered_data_value[~filtered_data_value['Area Code (M49)'].isin(codes_to_remove)]


# Filtrar el dataframe para eliminar las entradas con los códigos especificados
filtered_data_value = filtered_data_value[~filtered_data_value['Area Code (M49)'].isin(codes_to_remove)]

value_data = filtered_data_value[filtered_data_value['Element'] == 'Gross Production Value (constant 2014-2016 thousand I$)']

filtered_data_value


# Sumar la producción por país y por año
value_sum = value_data.groupby(['Area Code (M49)'])[year_columns].sum()
value_avg = value_sum.mean(axis=1)


# Función para contar las caídas por debajo del promedio
def count_falls(row, avg):
    return sum(value < avg for value in row)

# Aplicar la función para contar las caídas por debajo del promedio
value_falls_below_avg = value_sum.apply(lambda x: count_falls(x, value_avg[x.name]), axis=1).to_frame(name='value_falls_below_avg')

# Calcular el desvío estándar y la tendencia de la producción para cada país
value_std = (value_sum.std(axis=1)/value_avg).to_frame(name='value_std')
value_trend = (value_sum.apply(lambda x: stats.linregress(range(len(year_columns)), x)[0], axis=1)/value_avg).to_frame(name='value_trend')


# Calcular la diferencia interanual y contar caídas
value_diff = value_sum.diff(axis=1)
value_falls_interannual = (value_diff < 0).sum(axis=1).to_frame(name='value_falls_interannual ')



# Crear un DataFrame con las columnas deseadas

final_df= final_df.merge(value_falls_below_avg, left_on='Area Code (M49)', right_index=True, how='left')
final_df= final_df.merge(value_std, left_on='Area Code (M49)', right_index=True, how='left')
final_df=final_df.merge(value_trend, left_on='Area Code (M49)', right_index=True, how='left')
final_df=final_df.merge(value_falls_interannual, left_on='Area Code (M49)', right_index=True, how='left')
final_df['value_trend&std'] = final_df['value_trend'] * final_df['value_std']


final_df



In [ ]:
def shannon_diversity_index(group):
    total_yield = group['Y2021'].sum()
    if total_yield == 0:
        return 0
    proportions = group['Y2021'] / total_yield
    return -sum(p * np.log(p) for p in proportions if p > 0)

################ Calculando el índice de Shannon para cada área en 2021
diversity_2021 = yield_data.groupby('Alpha-3code').apply(shannon_diversity_index).reset_index()
diversity_2021 = diversity_2021.rename(columns={0: 'Shannon_Diversity_2021'})



############# Añadiendo el índice de Shannon a final_df
############# Asumiendo que 'Alpha-3code' es una columna en ambos DataFrames y es la clave para unirlos
final_df = final_df.merge(diversity_2021, left_on='Alpha-3code', right_on='Alpha-3code', how='left')


###################### calculo el indice de shannon promedio
def shannon_diversity_index(group):
    total_yield = group.sum()
    if total_yield == 0:
        return 0
    proportions = group / total_yield
    return -sum(p * np.log(p) for p in proportions if p > 0)

########## Asumiendo que las columnas de años están en el formato 'Y1992', 'Y1993', ..., 'Y2021'
years = [f'Y{year}' for year in range(1992, 2022)]

#############3 Calculando el índice de Shannon para cada año y área
diversity_indices = {}
for year in years:
    diversity_indices[year] = yield_data.groupby('Alpha-3code').apply(lambda x: shannon_diversity_index(x[year]))

############## Convirtiendo el diccionario en un DataFrame
diversity_df = pd.DataFrame(diversity_indices)

################ Calculando el promedio de los índices de Shannon para cada área
diversity_df['Shannon_Diversity_Average'] = diversity_df.mean(axis=1)

############ unirlo con final_df usando 'Alpha-3code'
final_df = final_df.merge(diversity_df[['Shannon_Diversity_Average']], left_on='Alpha-3code', right_index=True, how='left')
#final_df.head()
print(final_df.columns)


final_df

Index(['Area', 'Area Code (M49)', 'Alpha-2code', 'Alpha-3code',
       'falls_below_avg', 'falls_interannual', 'yield_std', 'yield_trend',
       'yield_trend&std', 'yield_flags_proportion', 'value_falls_below_avg',
       'value_std', 'value_trend', 'value_falls_interannual ',
       'value_trend&std', 'value_flags_proportion', 'Shannon_Diversity_2021',
       'Shannon_Diversity_Average'],
      dtype='object')


In [ ]:
############### EPXLORAR RELACION CON BIODIVERSIDAD!!!
############ Calcular la matriz de correlación
correlation_matrix = final_df[['Shannon_Diversity_Average', 'falls_below_avg', 'falls_interannual', 'yield_std', 'yield_trend', 'yield_trend&std', 'value_falls_below_avg',
       'value_std', 'value_trend', 'value_falls_interannual ',
       'value_trend&std']].corr()

################# Mostrar la matriz de correlación
print(correlation_matrix)


##################### Regresion

########## Definir la variable independiente
X = final_df['Shannon_Diversity_Average']  # Variable independiente

############ Definir las variables dependientes
Y = final_df[['falls_below_avg', 'falls_interannual', 'yield_std', 'yield_trend', 'yield_trend&std', 'value_falls_below_avg',
       'value_std', 'value_trend', 'value_falls_interannual ',
       'value_trend&std']]

##################### Añadir una constante a la variable independiente
X = sm.add_constant(X)

################ Crear y ajustar modelos de regresión para cada variable dependiente
for column in Y.columns:
  model = sm.OLS(Y[column], X).fit()
  print(f"Modelo para {column}:\n")
  print(model.summary())
  print("\n\n")



                           Shannon_Diversity_Average  falls_below_avg  \
Shannon_Diversity_Average                   1.000000        -0.068469   
falls_below_avg                            -0.068469         1.000000   
falls_interannual                          -0.129549        -0.072696   
yield_std                                  -0.057189         0.229306   
yield_trend                                 0.032784         0.198557   
yield_trend&std                            -0.079195         0.315017   
value_falls_below_avg                       0.051867         0.106540   
value_std                                  -0.071833         0.145319   
value_trend                                 0.035301         0.130988   
value_falls_interannual                    -0.027395        -0.161328   
value_trend&std                             0.019534         0.098309   

                           falls_interannual  yield_std  yield_trend  \
Shannon_Diversity_Average          -0.129549  -0.05

In [ ]:
# Filtrar final_df para eliminar filas con non_ax_flags_proportion_avg < 0.5
filtered_final_df = final_df[final_df['yield_flags_proportion']<= 0.5]


# Calcular la matriz de correlación para el conjunto de datos filtrado
correlation_matrix_filtered = filtered_final_df[['Shannon_Diversity_Average', 'falls_below_avg', 'falls_interannual', 'yield_std', 'yield_trend', 'yield_trend&std', 'value_falls_below_avg', 'value_std', 'value_trend', 'value_falls_interannual ',
       'value_trend&std']].corr()

# Mostrar la matriz de correlación
print(correlation_matrix_filtered)

# Definir la variable independiente para la regresión
X_filtered = filtered_final_df['Shannon_Diversity_Average']  # Variable independiente

# Definir las variables dependientes
Y_filtered = filtered_final_df[['falls_below_avg', 'falls_interannual', 'yield_std', 'yield_trend', 'yield_trend&std', 'value_falls_below_avg','value_std', 'value_trend', 'value_falls_interannual ',     'value_trend&std']]

# Añadir una constante a la variable independiente
X_filtered = sm.add_constant(X_filtered)

#############3 Crear y ajustar modelos de regresión para cada variable dependiente en el conjunto de datos filtrado
#for column in Y_filtered.columns:
#    model = sm.OLS(Y_filtered[column], X_filtered).fit()
#    print(f"Modelo para {column}:\n")
#    print(model.summary())
#    print("\n\n")
    
row_count = len(filtered_final_df1)
print(row_count)

                           Shannon_Diversity_Average  falls_below_avg  \
Shannon_Diversity_Average                   1.000000        -0.042519   
falls_below_avg                            -0.042519         1.000000   
falls_interannual                          -0.106062        -0.075521   
yield_std                                  -0.305226         0.189229   
yield_trend                                -0.227082         0.133537   
yield_trend&std                            -0.251424         0.190757   
value_falls_below_avg                      -0.023769         0.155739   
value_std                                   0.022689         0.406669   
value_trend                                 0.076409         0.223714   
value_falls_interannual                    -0.192284        -0.293585   
value_trend&std                             0.041564         0.250923   

                           falls_interannual  yield_std  yield_trend  \
Shannon_Diversity_Average          -0.106062  -0.30

In [ ]:
import numpy as np
from scipy import stats

# Cargar la base de datos GDP
gdp_data = pd.read_csv('input/gdp_worldbank.csv')

# Filtrar las columnas relevantes para el cálculo del promedio y la tendencia del PIB
gdp_data_filtered = gdp_data[['Country Code'] + [str(year) for year in range(1992, 2023)]]

# Calcular el promedio del PIB por país entre 1992 y 2022
gdp_mean = gdp_data_filtered.set_index('Country Code').mean(axis=1, skipna=True)

# Calcular la tendencia del PIB por país entre 1992 y 2022, solo si hay suficientes datos
def calculate_trend(row):
    valid_data = row.dropna()
    if len(valid_data) >= 10:
        return stats.linregress(range(1992, 1992 + len(valid_data)), valid_data.values)[0]
    else:
        return np.nan

gdp_trend = gdp_data_filtered.set_index('Country Code').apply(calculate_trend, axis=1)

# Crear un DataFrame con los resultados
gdp_stats = pd.DataFrame({'GDP_Mean': gdp_mean, 'GDP_Trend': gdp_trend}).reset_index()

# Unir los resultados con final_df usando 'Alpha-3code' como clave
final_df1 = final_df.merge(gdp_stats, left_on='Alpha-3code', right_on='Country Code', how='left')
final_df2 = final_df1.drop(columns=['Country Code'])
# Mostrar las primeras filas del DataFrame actualizado
final_df2.head()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt


# Cargar el mapa mundial
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


world = world.merge(final_df, left_on='iso_a3', right_on='Alpha-3code', how='left')
#world.head()
# Códigos en final_df que no están en world
missing_in_world = final_df[~final_df['Alpha-3code'].isin(world['iso_a3'])]['Alpha-3code'].unique()

# Códigos en world que no están en final_df
missing_in_final_df = world[~world['iso_a3'].isin(final_df['Alpha-3code'])]['iso_a3'].unique()

#print("Códigos en final_df que no están en world:", missing_in_world)
#print("Códigos en world que no están en final_df:", missing_in_final_df)

#world.to_file("world_yield.gpkg")





ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
